### DQ Dashboard - View Setup

Run once per environment (dev, uat, prod), same as `03_event_log_setup`. Creates the 3 views backing the DQ dashboard, scoped to the given `catalog` - a fresh, correctly-qualified set of views per environment, not one view trying to dynamically switch catalogs at query time.

Requires `03_event_log_setup` to have already been run for this catalog.

In [0]:
dbutils.widgets.text("catalog", "dev", "Target catalog")
catalog = dbutils.widgets.get("catalog")

### Expectations: Bronze and Silver, unified

Same query shape covers both layers - expectations are gathered in same place regardless of which pipeline wrote them.

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.stripex.dq_expectations_history AS
SELECT
  origin.pipeline_name,
  origin.flow_name AS table_name,
  timestamp,
  exp.col.name AS rule_name,
  exp.col.dataset,
  exp.col.passed_records,
  exp.col.failed_records,
  ROUND(exp.col.failed_records / NULLIF(exp.col.passed_records + exp.col.failed_records, 0) * 100, 2) AS failure_rate_pct
FROM {catalog}.stripex.ingestion_event_log_raw
CROSS JOIN LATERAL EXPLODE(
  FROM_JSON(details:flow_progress.data_quality.expectations,
    'array<struct<name:string, dataset:string, passed_records:int, failed_records:int>>')
) AS exp
WHERE event_type = 'flow_progress'
UNION ALL
SELECT
  origin.pipeline_name,
  origin.flow_name AS table_name,
  timestamp,
  exp.col.name AS rule_name,
  exp.col.dataset,
  exp.col.passed_records,
  exp.col.failed_records,
  ROUND(exp.col.failed_records / NULLIF(exp.col.passed_records + exp.col.failed_records, 0) * 100, 2) AS failure_rate_pct
FROM {catalog}.stripex.transformation_event_log_raw
CROSS JOIN LATERAL EXPLODE(
  FROM_JSON(details:flow_progress.data_quality.expectations,
    'array<struct<name:string, dataset:string, passed_records:int, failed_records:int>>')
) AS exp
WHERE event_type = 'flow_progress'
""")

print(f"Created {catalog}.stripex.dq_expectations_history")

### Quarantine-table monitoring, all 8 pairs (Bronze 7 and Silver 1)

Direct row-count queries against the tables themselves, not the event log - these tables exist independently of any single pipeline run.

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.stripex.dq_quarantine_counts AS
SELECT 'bronze_orders' AS source, COUNT(*) AS quarantined_count, current_timestamp() AS checked_at FROM {catalog}.stripex.bronze_orders_quarantined
UNION ALL SELECT 'bronze_order_items', COUNT(*), current_timestamp() FROM {catalog}.stripex.bronze_order_items_quarantined
UNION ALL SELECT 'bronze_customers', COUNT(*), current_timestamp() FROM {catalog}.stripex.bronze_customers_quarantined
UNION ALL SELECT 'bronze_products', COUNT(*), current_timestamp() FROM {catalog}.stripex.bronze_products_quarantined
UNION ALL SELECT 'bronze_categories', COUNT(*), current_timestamp() FROM {catalog}.stripex.bronze_categories_quarantined
UNION ALL SELECT 'bronze_clickstream', COUNT(*), current_timestamp() FROM {catalog}.stripex.bronze_clickstream_quarantined
UNION ALL SELECT 'bronze_inventory', COUNT(*), current_timestamp() FROM {catalog}.stripex.bronze_inventory_quarantined
UNION ALL SELECT 'silver_order_items', COUNT(*), current_timestamp() FROM {catalog}.stripex.silver_order_items_quarantined
""")

print(f"Created {catalog}.stripex.dq_quarantine_counts")

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.stripex.dq_referential_integrity AS
SELECT 'orders_to_customers' AS relationship, COUNT(*) AS orphaned_count, current_timestamp() AS checked_at
FROM {catalog}.stripex.silver_orders o
WHERE NOT EXISTS (SELECT 1 FROM {catalog}.stripex.silver_customers c WHERE c.customer_id = o.customer_id AND c.__END_AT IS NULL)
UNION ALL
SELECT 'order_items_to_orders', COUNT(*), current_timestamp()
FROM {catalog}.stripex.silver_order_items oi
WHERE NOT EXISTS (SELECT 1 FROM {catalog}.stripex.silver_orders o WHERE o.order_id = oi.order_id AND o.__END_AT IS NULL)
UNION ALL
SELECT 'order_items_to_products', COUNT(*), current_timestamp()
FROM {catalog}.stripex.silver_order_items oi
WHERE NOT EXISTS (SELECT 1 FROM {catalog}.stripex.silver_products p WHERE p.product_id = oi.product_id)
UNION ALL
SELECT 'products_to_categories', COUNT(*), current_timestamp()
FROM {catalog}.stripex.silver_products p
WHERE NOT EXISTS (SELECT 1 FROM {catalog}.stripex.silver_categories c WHERE c.category_id = p.category_id)
UNION ALL
SELECT 'inventory_to_products', COUNT(*), current_timestamp()
FROM {catalog}.stripex.silver_inventory i
WHERE NOT EXISTS (SELECT 1 FROM {catalog}.stripex.silver_products p WHERE p.product_id = i.product_id)
UNION ALL
SELECT 'clickstream_to_products', COUNT(*), current_timestamp()
FROM {catalog}.stripex.silver_clickstream ev
WHERE ev.product_id IS NOT NULL
  AND NOT EXISTS (SELECT 1 FROM {catalog}.stripex.silver_products p WHERE p.product_id = ev.product_id)
UNION ALL
SELECT 'clickstream_to_customers', COUNT(*), current_timestamp()
FROM {catalog}.stripex.silver_clickstream ev
WHERE ev.customer_id IS NOT NULL
  AND NOT EXISTS (SELECT 1 FROM {catalog}.stripex.silver_customers c WHERE c.customer_id = ev.customer_id AND c.__END_AT IS NULL)
UNION ALL
SELECT 'clickstream_to_orders', COUNT(*), current_timestamp()
FROM {catalog}.stripex.silver_clickstream ev
WHERE ev.order_id IS NOT NULL
  AND NOT EXISTS (SELECT 1 FROM {catalog}.stripex.silver_orders o WHERE o.order_id = ev.order_id AND o.__END_AT IS NULL)
""")

print(f"Created {catalog}.stripex.dq_referential_integrity")